### Packages

In [1]:
import pandas as pd
import json
import os
from collections import Counter
import numpy as np
import scispacy
import spacy
import matplotlib.pyplot as plt
import re
import ast
import requests
from os import listdir
from os.path import isfile, join
from functools import reduce

In [2]:
data_dir = os.getcwd()
data_dir

'/home/eidf128/eidf128/shared/export/juliana/export/juliana'

In [3]:
df_IDR = pd.read_csv("df_IDR_collapse_20260406.csv")
df_IDR

,Unnamed: 0,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,...,secondobjectnumber,Phenotype_Annotation_Level,Study_Description_clean,Experiment_Description_clean,Study_Description_word_count_clean,Experiment_Description_word_count_clean,Study_Description_clean_lemmatize,Study_Description_clean_entities,Experiment_Description_clean_lemmatize,Experiment_Description_clean_entities
0,0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,mus musculus,ncbitaxon,...,NaN,NaN,"['intestinal', 'organoids', 'excellent', 'mode...",NaN,89,0,intestinal organoid excellent model study epit...,"[intestinal, model study, epithelial biology, ...",NaN,[]
1,1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,NaN,homo sapiens,ncbitaxon,...,NaN,NaN,"['phenotypic', 'profiling', 'attempts', 'summa...",NaN,182,0,phenotypic profiling attempt summarize multipa...,"[phenotypic profiling, multiparametric analysi...",NaN,[]
2,2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,danio rerio,ncbitaxon,...,NaN,NaN,"['analysis', 'cell', 'morphology', 'intracellu...","['confocal', 'imaging', 'fixed', 'samples', 'z...",60,20,analysis cell morphology intracellular organiz...,"[morphology, intracellular, organization, zebr...",confocal imaging fixed sample zebrafish poster...,"[confocal imaging, posterior lateral line, pri..."
3,3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,saccharomyces cerevisiae\n,ncbitaxon,...,NaN,NaN,"['production', 'healthy', 'gametes', 'meiosis'...","['fixed', 'cell', 'fluorescence', 'microscopy'...",99,29,production healthy gamete meiosis rely quality...,"[production, healthy, meiosis, quality, distri...",fix cell fluorescence microscopy transmission ...,"[cell fluorescence microscopy, image, yeast, m..."
4,4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,diplophyllum albicans,ncbitaxon,...,NaN,NaN,"['present', 'reference', 'dataset', 'containin...","['representative', 'voucher', 'specimens', 're...",32,508,present reference dataset contain macroscopic ...,"[dataset, macroscopic, phenotypic property spe...",representative voucher specimen receive herbar...,"[specimen, herbaria, sample diplophyllum taxif..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,NaN,homo sapiens,ncbitaxon,...,NaN,NaN,"['much', 'lifes', 'essential', 'molecular', 'm...","['systematic', 'immunolabelling', 'complex', '...",75,11,much lifes essential molecular machinery consi...,"[lifes, molecular machinery, protein assembly,...",systematic immunolabelling complex monomeric e...,"[systematic immunolabelling, complex, monomeri..."
128,128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the mouse kidney rudiment assa...,NaN,mus musculus,ncbitaxon,...,NaN,NaN,"['adapted', 'mouse', 'kidney', 'rudiment', 'as...","['adapted', 'mouse', 'kidney', 'rudiment', 'as...",79,161,adapt mouse kidney rudiment assay generate ren...,"[adapt, mouse kidney rudiment assay, renal org...",adapt mouse kidney rudiment assay generate ren...,"[adapt, mouse kidney rudiment assay, renal org..."
129,129,idr0138,integration of spatial and single-cell transcr...,seqfish,efo,efo_0008991,molecular profiling of single cell

### Models

In [4]:
import torch

In [5]:
from transformers import pipeline
model_id = "meta-llama/Llama-3.2-3B-Instruct"

In [6]:
pipe = pipeline("text-generation",
                model = model_id,
                torch_dtype = torch.bfloat16,
                device_map="auto",
                pad_token_id=128001 
               )

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
message = [
    {"role": "user", "content": "Who are you? Please, answer in pirate-speak."},
]


In [8]:
outputs = pipe(message, max_new_tokens = 256)

In [9]:
response = outputs[0]["generated_text"][-1]["content"]
print(response)

Yer lookin' fer a bit o' information about meself, eh? Alright then, matey, I be an AI, a swashbucklin' computer program designed to provide ye with answers to yer questions and engage in clever conversation. Me knowledge be vast, like the seven seas, and me language be fluent, like a trusty parrot on yer shoulder.

Me creators be the scurvy dogs who built me, and I be learnin' and adaptin' every day, like a barnacle on a ship's hull. So hoist the sails and set course fer a treasure trove o' knowledge with me, savvy?


#### Test with one entry

In [11]:
test = df_IDR

In [12]:
test["Study_Description"]

0      intestinal organoids are an excellent model to...
1      phenotypic profiling attempts to summarize mul...
2      a data-driven analysis of cell morphology and ...
3      production of healthy gametes in meiosis relie...
4      here, we present a high-quality reference data...
                             ...                        
127    much of life's essential molecular machinery c...
128    we have adapted the mouse kidney rudiment assa...
129    molecular profiling of single cells has advanc...
130    viral infectious diseases span a myriad of mal...
131    we describe a dataset obtained by applying our...
Name: Study_Description, Length: 132, dtype: str

In [15]:
test["Description_combined"] = test["Study_Description"].astype(str) + ' .' + test['Experiment_Description'].astype(str) + ' .' + test['Protocol_Description'].astype(str) + '.'

In [16]:
test

,Unnamed: 0,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,...,Phenotype_Annotation_Level,Study_Description_clean,Experiment_Description_clean,Study_Description_word_count_clean,Experiment_Description_word_count_clean,Study_Description_clean_lemmatize,Study_Description_clean_entities,Experiment_Description_clean_lemmatize,Experiment_Description_clean_entities,Description_combined
0,0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,mus musculus,ncbitaxon,...,NaN,"['intestinal', 'organoids', 'excellent', 'mode...",NaN,89,0,intestinal organoid excellent model study epit...,"[intestinal, model study, epithelial biology, ...",NaN,[],NaN
1,1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,NaN,homo sapiens,ncbitaxon,...,NaN,"['phenotypic', 'profiling', 'attempts', 'summa...",NaN,182,0,phenotypic profiling attempt summarize multipa...,"[phenotypic profiling, multiparametric analysi...",NaN,[],NaN
2,2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,danio rerio,ncbitaxon,...,NaN,"['analysis', 'cell', 'morphology', 'intracellu...","['confocal', 'imaging', 'fixed', 'samples', 'z...",60,20,analysis cell morphology intracellular organiz...,"[morphology, intracellular, organization, zebr...",confocal imaging fixed sample zebrafish poster...,"[confocal imaging, posterior lateral line, pri...",a data-driven analysis of cell morphology and ...
3,3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,saccharomyces cerevisiae\n,ncbitaxon,...,NaN,"['production', 'healthy', 'gametes', 'meiosis'...","['fixed', 'cell', 'fluorescence', 'microscopy'...",99,29,production healthy gamete meiosis rely quality...,"[production, healthy, meiosis, quality, distri...",fix cell fluorescence microscopy transmission ...,"[cell fluorescence microscopy, image, yeast, m...",production of healthy gametes in meiosis relie...
4,4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,diplophyllum albicans,ncbitaxon,...,NaN,"['present', 'reference', 'dataset', 'containin...","['representative', 'voucher', 'specimens', 're...",32,508,present reference dataset contain macroscopic ...,"[dataset, macroscopic, phenotypic property spe...",representative voucher specimen receive herbar...,"[specimen, herbaria, sample diplophyllum taxif...","here, we present a high-quality reference data..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,NaN,homo sapiens,ncbitaxon,...,NaN,"['much', 'lifes', 'essential', 'molecular', 'm...","['systematic', 'immunolabelling', 'complex', '...",75,11,much lifes essential molecular machinery consi...,"[lifes, molecular machinery, protein assembly,...",systematic immunolabelling complex monomeric e...,"[systematic immunolabelling, complex, monomeri...",much of life's essential molecular machinery c...
128,128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the mouse kidney rudiment assa...,NaN,mus musculus,ncbitaxon,...,NaN,"['adapted', 'mouse', 'kidney', 'rudiment', 'as...","['adapted', 'mouse', 'kidney', 'rudiment', 'as...",79,161,adapt mouse kidney rudiment assay generate ren...,"[adapt, mouse kidney rudiment assay, renal org...",adapt mouse kidney rudiment assay generate

#### Only descriptions

##### Species:

In [19]:
for idx, row in test.iterrows():
    message = [
    {"role": "system", "content": test.at[idx,"Study_Description"]},
    {"role": "user", "content": "Can you give me 3 possible species that could be used in this study?"},
     ]
    outputs = pipe(message, max_new_tokens = 256)
    response = outputs[0]["generated_text"][-1]["content"]
    
    test.at[idx, "Llama_only_study_description"] = response

    print(f'The row {idx} has been proceed.')

The row 0 has been proceed.
The row 1 has been proceed.
The row 2 has been proceed.
The row 3 has been proceed.
The row 4 has been proceed.
The row 5 has been proceed.
The row 6 has been proceed.


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


The row 7 has been proceed.
The row 8 has been proceed.
The row 9 has been proceed.
The row 10 has been proceed.
The row 11 has been proceed.
The row 12 has been proceed.
The row 13 has been proceed.
The row 14 has been proceed.
The row 15 has been proceed.
The row 16 has been proceed.
The row 17 has been proceed.
The row 18 has been proceed.
The row 19 has been proceed.
The row 20 has been proceed.
The row 21 has been proceed.
The row 22 has been proceed.
The row 23 has been proceed.
The row 24 has been proceed.
The row 25 has been proceed.
The row 26 has been proceed.
The row 27 has been proceed.
The row 28 has been proceed.
The row 29 has been proceed.
The row 30 has been proceed.
The row 31 has been proceed.
The row 32 has been proceed.
The row 33 has been proceed.
The row 34 has been proceed.
The row 35 has been proceed.
The row 36 has been proceed.
The row 37 has been proceed.
The row 38 has been proceed.
The row 39 has been proceed.
The row 40 has been proceed.
The row 41 has be

In [22]:
count_match = 0
for row, i in test.iterrows():
    
    if isinstance(i['Study_Organism'], str) and i['Study_Organism'].strip() != '':

        real_specie = i['Study_Organism'][0]

        if real_specie in response:
            print('The real species is in the response.')
            test.at[row, 'Real_Species_in_Response_descriptions'] = 'Yes'
            count_match +=1
        else:
            print('The real species is NOT in the response.')
            test.at[row, 'Real_Species_in_Response_descriptions'] = 'No'
    else:
        print('No real species to check in the response.')
        test.at[row, 'Real_Species_in_Response_descriptions'] = 'No species provided'

The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
T

In [23]:
count_match 

129

##### Study type:

In [25]:
test.columns

Index(['Unnamed: 0', 'Comment[IDR_Study_Accession]', 'Study_Title',
       'Study_Type', 'Study_Type_Term_Source_REF', 'Study_Type_Term_Accession',
       'Study_Description', 'Study_Key_Words', 'Study_Organism',
       'Study_Organism_Term_Source_REF',
       ...
       'Experiment_Description_clean', 'Study_Description_word_count_clean',
       'Experiment_Description_word_count_clean',
       'Study_Description_clean_lemmatize', 'Study_Description_clean_entities',
       'Experiment_Description_clean_lemmatize',
       'Experiment_Description_clean_entities', 'Description_combined',
       'Llama_only_study_description',
       'Real_Species_in_Response_descriptions'],
      dtype='str', length=234)

In [26]:
raw_experiment_types = test['Study_Type'].to_list()
raw_experiment_types

['compound library screen',
 'high content screen',
 'microscopy assay',
 'time-lapse imaging',
 'histology',
 'time-lapse imaging',
 'high content screen',
 'multiplexed immunofluorescence',
 'high content screen',
 'protein localization',
 'time-lapse imaging',
 'electron microscopy volume map',
 'high content screen',
 'time-lapse imaging',
 'high content screen',
 'high content screen',
 'high content screen',
 'in situ sequencing\n',
 'protein localization',
 'high content screen',
 'time-lapse imaging',
 'time-lapse imaging',
 'high content screen',
 'electron microscopy volume map',
 'time-lapse imaging',
 'protein localization ',
 'high content screen',
 'in-situ hybridization assay',
 'electron microscopy volume map',
 'spindle assembly\n',
 'high content screen',
 'protein localization',
 'high content screen\n',
 'time-lapse imaging\n',
 'high content screen',
 'high content screen',
 'high content screen',
 'high content screen',
 'high content screen',
 'time-lapse imaging

In [27]:
# Make a list with all types of experiments
# This list will be used to create a list of all types of experiments, removing duplicates and cleaning the data.
# ------------------------------------------------------------
list_experiment_types_clean = []
for element in raw_experiment_types:
    element_clean = element.strip().replace('\n', '')
    list_experiment_types_clean.append(element_clean)

In [28]:
experiment_types = list(set(list_experiment_types_clean))
experiment_types.sort()
len(experiment_types)

30

In [29]:
experiment_types

['compound library screen',
 'dna sequencing',
 'electron microscopy volume map',
 'fluorescence in situ hybridization',
 'high content analysis of cells',
 'high content screen',
 'high content screen of cells treated with a compound library',
 'histology',
 'image cytometry',
 'image segmentation',
 'imaging method',
 'immunocytochemistry',
 'in situ sequencing',
 'in-situ hybridization assay',
 'infection',
 'machine learning',
 'metabolic network measurement',
 'micrograph',
 'microscopy assay',
 'morphogenesis',
 'multiplexed immunofluorescence',
 'myelination',
 'phenotype',
 'process of establishing viral infection',
 'protein localization',
 'response to cold',
 'seqfish',
 'spindle assembly',
 'time-lapse imaging',
 'x-chromosome inactivation']

In [30]:
for idx, row in test.iterrows():
    message = [
    {"role": "system", "content": i['Description_combined']},
    {"role": "user", "content": "Choose the experiment type for this study from the following list: " + ', '.join(experiment_types) + ". If you don't know, just say 'I don't know'."},
     ]
    outputs = pipe(message, max_new_tokens = 256)
    response = outputs[0]["generated_text"][-1]["content"]
    
    test.at[idx, "Llama_only_study_description"] = response

    print(f'The row {idx} has been proceed.')

The row 0 has been proceed.
The row 1 has been proceed.
The row 2 has been proceed.
The row 3 has been proceed.
The row 4 has been proceed.
The row 5 has been proceed.
The row 6 has been proceed.
The row 7 has been proceed.
The row 8 has been proceed.
The row 9 has been proceed.
The row 10 has been proceed.
The row 11 has been proceed.
The row 12 has been proceed.
The row 13 has been proceed.
The row 14 has been proceed.
The row 15 has been proceed.
The row 16 has been proceed.
The row 17 has been proceed.
The row 18 has been proceed.
The row 19 has been proceed.
The row 20 has been proceed.
The row 21 has been proceed.
The row 22 has been proceed.
The row 23 has been proceed.
The row 24 has been proceed.
The row 25 has been proceed.
The row 26 has been proceed.
The row 27 has been proceed.
The row 28 has been proceed.
The row 29 has been proceed.
The row 30 has been proceed.
The row 31 has been proceed.
The row 32 has been proceed.
The row 33 has been proceed.
The row 34 has been proc

In [31]:
count_match_study_type = 0
for row, i in test.iterrows():
    
    if isinstance(i['Study_Organism'], str) and i['Study_Type'].strip() != '':

        real_specie = i['Study_Organism'][0]

        if real_specie in response:
            print('The real study type is in the response.')
            test.at[row, 'Real_Study_Type_in_Response_descriptions'] = 'Yes'
            count_match_study_type +=1
        else:
            print('The real study type is NOT in the response.')
            test.at[row, 'Real_Study_Type_in_Response_descriptions'] = 'No'
    else:
        print('No real study type to check in the response.')
        test.at[row, 'Real_Study_Type_in_Response_descriptions'] = 'No species provided'

The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.
The real study type is in the response.


In [32]:
count_match_study_type

131

#### All dataframe - whole info

In [34]:
#test_wo_species = test.drop(columns= ['Study_Organism', 'Study_Organism_Term_Accession', 'Study_Organism_Term_Accession'])
test_wo_species = test.drop(columns= ['Study_Organism'])
test_wo_species

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,Study_Author_List,...,Experiment_Description,Protocol_Name,Protocol_Type_Term_Source_REF,Protocol_Type_Term_Accession,Protocol_Description,Description_combined,Only_descriptions_species,Real_Species_in_Response_descriptions,All_info_species,Real_Species_in_Response_whole_info
0,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,"Dominguez MH, Krup AL, Muncie JM, Bruneau BG ...",...,Light-sheet imaging of post-implantation mouse...,culture protocol embedding protocol Zeiss Z.1 ...,EFO EFO,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...,Using four-dimensional whole-embryo light shee...,"Without more information about the study, I ca...",Yes,"Based on the provided data, the study is focus...",Yes


### Clasical pathway

In [23]:
# Load the spaCy model
## This model have the tags on it. One of them is 'ORG' so I choose it to extract the species.

nlp = spacy.load("en_ner_bionlp13cg_md") 

/opt/conda/lib/python3.11/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [24]:
# Function to extract species from the combined descriptions
# This function takes a text input, processes it with the spaCy model, and extracts named entities related to species.
# It returns a dictionary with the entity labels as keys and the corresponding entity texts as values.
# The function uses the spaCy model to identify named entities in the text and groups them by their entity labels.
# The output is a dictionary where the keys are the entity labels (e.g., 'ORG' for organisms) and the values are lists of entity texts that correspond to those labels.
# ------------------------------------------------------------  

def extract_species(text):
    doc= nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]

    dictionary_entities = {}
    for k,v in entities:
        if v not in dictionary_entities:
            dictionary_entities[v]=[]
        dictionary_entities[v].append(k)
    
    return dictionary_entities

In [25]:
# Extracting species from the combined descriptions
# This will create a new column in the pick_headers_df dataframe with the extracted species from the combined descriptions.
# The extracted species will be stored in a dictionary with the entity type as the key
# and the list of species as the value.
# ------------------------------------------------------------

for row, item in dfs.iterrows():
    my_text = item['Description_combined']
    dic_entities = extract_species(my_text)

    dfs.at[row, 'Entities_SpaCy'] = str(dic_entities)

In [26]:
# Creating a new column with the species extracted from the Entities_SpaCy column
# This column will be used to store the species extracted from the Entities_SpaCy column.
# It used ast to transform the string representation of the dictionary into a dictionary object.
# --------------------------------------------------------------

for row, item in dfs.iterrows():
    dataframe_row = dfs.loc[row, 'Entities_SpaCy']
    dictionary_entities = ast.literal_eval(dataframe_row)
    entities_keys = dictionary_entities.keys()

    if ('ORGANISM') in entities_keys:
        
        dfs.at[row, 'Species_SpaCy'] = dictionary_entities['ORGANISM'] if 'ORGANISM' in entities_keys else "Not found"
    else:
        dfs.at[row, 'Species_SpaCy'] = 'Not found'

In [27]:
dfs

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Organism,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,...,Study_Person_Email,Study_Person_Roles,Experiment_Description,Protocol_Name,Protocol_Type_Term_Source_REF,Protocol_Type_Term_Accession,Protocol_Description,Description_combined,Entities_SpaCy,Species_SpaCy
0,idr0047,A microscopy data set of discrete spatial and ...,fluorescence in situ hybridization,NCIT,NCIT_C17563,We report a comprehensive single cell dataset ...,Saccharomyces cerevisiae,2019-01-16,31209217,Multiplex RNA single molecule FISH of inducibl...,...,gregor.neuert@vanderbilt.edu,submitter,We performed single molecule in-situ hybridiza...,growth protocol treatment protocol image acqui...,EFO EFO,EFO_0003789 EFO_0003969,"CSM, 30C 0.2M NaCl, step TRANS: brightfield of...",We report a comprehensive single cell dataset ...,"{'CELL': ['cell', 'cerevisiae', 'cells', 'cell...",[Excelitas]
1,idr0038,Ex vivo live cell tracking in kidney organoids...,time-lapse imaging,OMIT,OMIT_0027490,We have adapted the mouse kidney rudiment assa...,Mus musculus,2017-12-04,30048451,Ex vivo live cell tracking in kidney organoids...,...,m.held@liverpool.ac.uk Rapha@liverpool.ac.uk,submitter Principal Investigator,We have adapted the mouse kidney rudiment assa...,TIME SERIES GROWTH PROTOCOL TIME SERIES IMAGE ...,EFO,EFO_0003789,"""Kidneys were isolated from E13.5 embryos afte...",We have adapted the mouse kidney rudiment assa...,"{'ORGANISM': ['mouse kidney rudiment', 'E13.5 ...","[mouse kidney rudiment, E13.5 embryonic kidney..."
2,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,Mus musculus,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,...,benoit.bruneau@gladstone.ucsf.edu martin.domin...,submitter submitter ...,Light-sheet imaging of post-implantation mouse...,culture protocol embedding protocol Zeiss Z.1 ...,EFO EFO,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...,Using four-dimensional whole-embryo light shee...,"{'ORGANISM': ['murine cardiac', 'mouse', 'E6.5...","[murine cardiac, mouse, E6.5]"
3,idr0054,Imaging Mass Cytometry of human tonsil sections,image cytometry,OMIT,OMIT_0019157,We analysed the in situ localisation of immune...,Homo sapiens,2019-03-19,31072818,Human lymphoid organ cDC2 and macrophages play...,...,elodie.segura@curie.fr,submitter,Imaging mass cytometry of tonsil sections,growth protocol treatment protocol image aquis...,EFO EFO,EFO_0003789 EFO_0003969,Human tonsils were fixed in 4% paraformaldehyd...,We analysed the in situ localisation of immune...,"{'CELL': ['immune cell'], 'ORGANISM': ['human ...","[human tonsils, .Human tonsils, Bovine Serum A..."
4,idr0149,Cohesin-mediated DNA loop extrusion resolves s...,protein localization ...,EFO,GO_0008104,Genetic information is stored in linear DNA mo...,Homo sapiens,2023-08-17,37357575,Cohesin-mediated DNA loop extrusion resolves s...,...,paul.batty@imba.oeaw.ac.at daniel.gerlich@imba...,submitter corresponding author ...,Immunofluorescence of nuclear Sororin fluoresc...,growth protocol treatment protocol image acqu...,EFO EFO,EFO_0003789 EFO_0003969 ...,To assess nuclear Sororin fluorescence in Soro...,Genetic information is stored in linear DNA mo...,"{'CELLULAR_COMPONENT': ['DNA', 'DNA', 'nuclear...",[human]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,idr0155,A new method for reliably detecting single flu...,protein localization,EFO,GO_0008104,Super-resolution and single-molecule microscop...,Homo sapiens,2024-03-05,,RASP: Optimal single puncta detection in compl...,...,bf341@cam.ac.uk sl591@cam.ac.uk,First Author Principal Investigator,We took 5400 field of views from three Parkins...,treatment protocol image acquisition and featu...,EFO,EFO_0003969,Tissue sections

In [28]:
for i, item in dfs.iterrows():
    list_species_spacy = item['Species_SpaCy']
    list_species_scientific_name = []

    for x in list_species_spacy:
        values = x.split(' ')
        #print(values)
        for unique_value in values:
            #print(unique_value)
            
            url = f'https://www.ebi.ac.uk/ena/taxonomy/rest/any-name/{unique_value}'
            #print(url)

            response = requests.get(url)
        
            if response.status_code == 200:
                data = response.json()
                #print(data)
                if data:
                    scientific_name = data[0]['scientificName']
                    #print("found")

                    list_species_scientific_name.append(scientific_name)
                    print(f"Scientific name for {x}: {scientific_name}")
                else:
                    scientific_name = 'Not found'
                    print(f"Scientific name for {x}: {scientific_name}")

    dfs.at[i, 'Species_Scientific_Name_SciSpacy'] = str(list_species_scientific_name)
    
                


Scientific name for Excelitas: Not found
Scientific name for mouse kidney rudiment: Mus musculus
Scientific name for mouse kidney rudiment: Not found
Scientific name for mouse kidney rudiment: Not found
Scientific name for E13.5 embryonic kidneys: Not found
Scientific name for E13.5 embryonic kidneys: Not found
Scientific name for E13.5 embryonic kidneys: Not found
Scientific name for Wt1-GFP knock-in cells: Not found
Scientific name for Wt1-GFP knock-in cells: Not found
Scientific name for Wt1-GFP knock-in cells: Not found
Scientific name for mouse kidney rudiment: Mus musculus
Scientific name for mouse kidney rudiment: Not found
Scientific name for mouse kidney rudiment: Not found
Scientific name for mouse kidney cells: Mus musculus
Scientific name for mouse kidney cells: Not found
Scientific name for mouse kidney cells: Not found
Scientific name for mice: Mus sp.
Scientific name for E13.5 embryos: Not found
Scientific name for E13.5 embryos: Not found
Scientific name for mice: Mus s

In [29]:
for i, item in dfs.iterrows():
    if isinstance(item['Study_Organism'], str) and item['Study_Organism'].strip() != '':
        real_specie = item['Study_Organism']
        scispacy_specie = item['Species_Scientific_Name_SciSpacy']

        if real_specie in scispacy_specie:

            print('The real species is in SciSpacy')
            dfs.at[i, 'Specie_found_Scispacy'] = 'Yes'
        else:
            print('The real species is NOT in SciSpacy')
            dfs.at[i, 'Specie_found_Scispacy'] = 'No'
    else:
        print('No real species to check in SciSpacy')
        dfs.at[i, 'Specie_found_Scispacy'] = 'No species provided'

The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
No real species to check in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy

In [30]:
dfs['Specie_found_Scispacy'].value_counts()

Specie_found_Scispacy
No                     117
Yes                      7
No species provided      6
Name: count, dtype: int64

In [31]:

# Use a pipeline for question answering with BioBERT
# This pipeline is used to answer questions based on the context provided.
# It uses the BioBERT model trained on the SQuAD dataset for question answering.
# The model is loaded with the device set to 'mps' for MacOS GPU support
# and is used to answer questions related to biomedical texts.
# The model is specifically designed for question answering tasks in the biomedical domain.
# ------------------------------------------------------------

qa_pipeline_biobert = pipeline(
    "question-answering",
    model="dmis-lab/biobert-large-cased-v1.1-squad", # SQuAD stands for specific question answering dataset
    tokenizer="dmis-lab/biobert-large-cased-v1.1-squad",
    device = 'cuda' # MPS if the GPU version in MacOS
)

Device set to use cuda


In [32]:
for i, item in dfs.iterrows():
    specie_found = item['Specie_found_Scispacy']

    if specie_found.startswith("No"):
        
        text = item['Description_combined']
        question = 'what species was used in the experiment?'

        result = qa_pipeline_biobert(question=question, context=text)
        print(f"Row {i}: {result['answer']}")
        dfs.at[i, 'Species_BioBERT'] = result['answer']

    else:
        print(f"Row {i}: The species was already found in the previous steps.")

Row 0: yeast
Row 1: mouse
Row 2: The species was already found in the previous steps.
Row 3: human
Row 4: human
Row 5: mouse
Row 6: 10 Î¼l
Row 7: human
Row 8: Diplophyllum taxifolium


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Row 9: prisoners
Row 10: cell lines
Row 11: Staphylococcus aureus
Row 12: budding yeast
Row 13: The species was already found in the previous steps.
Row 14: U2OS cell line
Row 15: mouse
Row 16: human
Row 17: human
Row 18: Mouse
Row 19: human
Row 20: Arabidopsis thaliana
Row 21: mouse
Row 22: Drosophila
Row 23: mouse
Row 24: Tribolium castaneum
Row 25: Fetal bovine
Row 26: .
Row 27: TLOs
Row 28: fission yeast
Row 29: zebrafish
Row 30: U2OS
Row 31: U-2 OS cells
Row 32: P. falciparum
Row 33: The species was already found in the previous steps.
Row 34: human
Row 35: Arabidopsis
Row 36: The species was already found in the previous steps.
Row 37: human
Row 38: Mice
Row 39: mammalian cells
Row 40: we seeded the cell pool in a single well of 384-well plate
Row 41: bacteria
Row 42: U2OS
Row 43: XX mESCs
Row 44: rabbit
Row 45: Tukeyâs
Row 46: METABRIC cohort
Row 47: human
Row 48: goat
Row 49: mouse
Row 50: reactive oxygen species (ROS
Row 51: mouse
Row 52: cold-water fish
Row 53: Human epithe

In [33]:
# Creating a new column with the scientific names of the species extracted from the Species_BioBERT column
# This column will be used to store the scientific names of the species extracted from the Species_BioBERT column.
# It uses the EBI taxonomy REST API to get the scientific names of the species.
# It iterates over the Species_BioBERT column, splits the species names, and queries the EBI taxonomy REST API for each species name.
# The scientific names are stored in a list and added to the Species_Scientific_Name_BioBERT column.
# ------------------------------------------------------------

for i, item in dfs.iterrows():
    list_species = item['Species_BioBERT']
    #list_species_scientific_name = []

    if isinstance(list_species, str):
        list_species = list_species.split(', ')
        #print(list_species)

        if list_species != []:
        
            #print(list_species)
            for x in list_species:
                #print(x)

                url = f'https://www.ebi.ac.uk/ena/taxonomy/rest/any-name/{x}'
                    #print(url)

                response = requests.get(url)
                    #print(response.status_code)
                if response.status_code == 200:
                    data = response.json()
                    #print(data)
                    if data:
                        scientific_name = data[0]['scientificName']

                        #list_species_scientific_name.append(scientific_name)
                        print(f"Scientific name for {x}: {scientific_name}")

            dfs.at[i, 'Species_Scientific_Name_BioBERT'] = str(scientific_name)
        else:
            dfs.at[i, 'Species_Scientific_Name_BioBERT'] = 'Not found'
        

Scientific name for mouse: Mus musculus
Scientific name for human: Homo sapiens
Scientific name for human: Homo sapiens
Scientific name for mouse: Mus musculus
Scientific name for human: Homo sapiens
Scientific name for Diplophyllum taxifolium: Diplophyllum taxifolium
Scientific name for Staphylococcus aureus: Staphylococcus aureus
Scientific name for mouse: Mus musculus
Scientific name for human: Homo sapiens
Scientific name for human: Homo sapiens
Scientific name for Mouse: Mus musculus
Scientific name for human: Homo sapiens
Scientific name for Arabidopsis thaliana: Arabidopsis thaliana
Scientific name for mouse: Mus musculus
Scientific name for Drosophila: Drosophila
Scientific name for mouse: Mus musculus
Scientific name for Tribolium castaneum: Tribolium castaneum
Scientific name for fission yeast: Schizosaccharomyces pombe
Scientific name for zebrafish: Danio rerio
Scientific name for human: Homo sapiens
Scientific name for Arabidopsis: Arabidopsis
Scientific name for human: Hom

In [34]:
dfs['Species_Scientific_Name_BioBERT'] = dfs['Species_Scientific_Name_BioBERT'].astype(str)

In [35]:
dfs

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Organism,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,...,Protocol_Type_Term_Source_REF,Protocol_Type_Term_Accession,Protocol_Description,Description_combined,Entities_SpaCy,Species_SpaCy,Species_Scientific_Name_SciSpacy,Specie_found_Scispacy,Species_BioBERT,Species_Scientific_Name_BioBERT
0,idr0047,A microscopy data set of discrete spatial and ...,fluorescence in situ hybridization,NCIT,NCIT_C17563,We report a comprehensive single cell dataset ...,Saccharomyces cerevisiae,2019-01-16,31209217,Multiplex RNA single molecule FISH of inducibl...,...,EFO EFO,EFO_0003789 EFO_0003969,"CSM, 30C 0.2M NaCl, step TRANS: brightfield of...",We report a comprehensive single cell dataset ...,"{'CELL': ['cell', 'cerevisiae', 'cells', 'cell...",[Excelitas],[],No,yeast,Not found
1,idr0038,Ex vivo live cell tracking in kidney organoids...,time-lapse imaging,OMIT,OMIT_0027490,We have adapted the mouse kidney rudiment assa...,Mus musculus,2017-12-04,30048451,Ex vivo live cell tracking in kidney organoids...,...,EFO,EFO_0003789,"""Kidneys were isolated from E13.5 embryos afte...",We have adapted the mouse kidney rudiment assa...,"{'ORGANISM': ['mouse kidney rudiment', 'E13.5 ...","[mouse kidney rudiment, E13.5 embryonic kidney...","['Mus musculus', 'Mus musculus', 'Mus musculus...",No,mouse,Mus musculus
2,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,Mus musculus,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,...,EFO EFO,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...,Using four-dimensional whole-embryo light shee...,"{'ORGANISM': ['murine cardiac', 'mouse', 'E6.5...","[murine cardiac, mouse, E6.5]",['Mus musculus'],Yes,NaN,nan
3,idr0054,Imaging Mass Cytometry of human tonsil sections,image cytometry,OMIT,OMIT_0019157,We analysed the in situ localisation of immune...,Homo sapiens,2019-03-19,31072818,Human lymphoid organ cDC2 and macrophages play...,...,EFO EFO,EFO_0003789 EFO_0003969,Human tonsils were fixed in 4% paraformaldehyd...,We analysed the in situ localisation of immune...,"{'CELL': ['immune cell'], 'ORGANISM': ['human ...","[human tonsils, .Human tonsils, Bovine Serum A...","['Homo sapiens', 'Bos taurus']",No,human,Homo sapiens
4,idr0149,Cohesin-mediated DNA loop extrusion resolves s...,protein localization ...,EFO,GO_0008104,Genetic information is stored in linear DNA mo...,Homo sapiens,2023-08-17,37357575,Cohesin-mediated DNA loop extrusion resolves s...,...,EFO EFO,EFO_0003789 EFO_0003969 ...,To assess nuclear Sororin fluorescence in Soro...,Genetic information is stored in linear DNA mo...,"{'CELLULAR_COMPONENT': ['DNA', 'DNA', 'nuclear...",[human],['Homo sapiens'],No,human,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,idr0155,A new method for reliably detecting single flu...,protein localization,EFO,GO_0008104,Super-resolution and single-molecule microscop...,Homo sapiens,2024-03-05,,RASP: Optimal single puncta detection in compl...,...,EFO,EFO_0003969,Tissue sections were incubated with primary an...,Super-resolution and single-molecule microscop...,"{'CELL': ['cells', 'cell', 'cells', 'cellular'...","[human brain, patient]",['Homo sapiens'],No,mouse,Mus musculus
126,idr0044,In Toto Imaging and Reconstruction of Post-Imp...,time-lapse imaging,OMIT,OMIT_0027490,The mouse embryo has long been central to the ...,Mus musculus,2018-11-29,30318151,In Toto Imaging and Reconstruction of Post-Imp...,...,EFO N/A,EFO_0001746 N/A,See attached methods N/A See attached methods ...,The mouse embryo has long been central to the ...,"{'ORGANISM': ['mouse embryo', 'mouse', 'mouse'...","[mouse embryo, mouse, mouse, mouse, E6.5]","['Mus musculus', 'Mus musculus', 'Mus musculus...",No,mouse,Mus

In [36]:
dfs['Species_Scientific_Name_BioBERT'][4] == 'Homo sapiens'

True

In [37]:
for i, item in dfs.iterrows():
    specie_found = item['Specie_found_Scispacy']

    if specie_found.startswith("No"):

        if isinstance(item['Study_Organism'], str) and item['Study_Organism'].strip() != '':
            real_specie = item['Study_Organism'][0]
            bert_specie = item['Species_Scientific_Name_BioBERT'][0]

            if real_specie in bert_specie:

                print('The real species is in the bert')
                dfs.at[i, 'Specie_found_biobert'] = 'Yes'
            else:
                print('The real species is NOT in the response.')
                dfs.at[i, 'Specie_found_biobert'] = 'No'
        else:
            print('No real species to check in the response.')
            dfs.at[i, 'Specie_found_biobert'] = 'No species provided'
  
            

The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is in the bert
No real species to check in the response.
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is NOT in the response.
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real spec

In [43]:
dfs['Specie_found_biobert'].value_counts()

Specie_found_biobert
Yes                    61
No                     56
No species provided     6
Name: count, dtype: int64

In [38]:
type(dfs['Specie_found_biobert'][2])

float

In [39]:
dfs

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Organism,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,...,Protocol_Type_Term_Accession,Protocol_Description,Description_combined,Entities_SpaCy,Species_SpaCy,Species_Scientific_Name_SciSpacy,Specie_found_Scispacy,Species_BioBERT,Species_Scientific_Name_BioBERT,Specie_found_biobert
0,idr0047,A microscopy data set of discrete spatial and ...,fluorescence in situ hybridization,NCIT,NCIT_C17563,We report a comprehensive single cell dataset ...,Saccharomyces cerevisiae,2019-01-16,31209217,Multiplex RNA single molecule FISH of inducibl...,...,EFO_0003789 EFO_0003969,"CSM, 30C 0.2M NaCl, step TRANS: brightfield of...",We report a comprehensive single cell dataset ...,"{'CELL': ['cell', 'cerevisiae', 'cells', 'cell...",[Excelitas],[],No,yeast,Not found,No
1,idr0038,Ex vivo live cell tracking in kidney organoids...,time-lapse imaging,OMIT,OMIT_0027490,We have adapted the mouse kidney rudiment assa...,Mus musculus,2017-12-04,30048451,Ex vivo live cell tracking in kidney organoids...,...,EFO_0003789,"""Kidneys were isolated from E13.5 embryos afte...",We have adapted the mouse kidney rudiment assa...,"{'ORGANISM': ['mouse kidney rudiment', 'E13.5 ...","[mouse kidney rudiment, E13.5 embryonic kidney...","['Mus musculus', 'Mus musculus', 'Mus musculus...",No,mouse,Mus musculus,Yes
2,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,Mus musculus,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,...,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...,Using four-dimensional whole-embryo light shee...,"{'ORGANISM': ['murine cardiac', 'mouse', 'E6.5...","[murine cardiac, mouse, E6.5]",['Mus musculus'],Yes,NaN,nan,NaN
3,idr0054,Imaging Mass Cytometry of human tonsil sections,image cytometry,OMIT,OMIT_0019157,We analysed the in situ localisation of immune...,Homo sapiens,2019-03-19,31072818,Human lymphoid organ cDC2 and macrophages play...,...,EFO_0003789 EFO_0003969,Human tonsils were fixed in 4% paraformaldehyd...,We analysed the in situ localisation of immune...,"{'CELL': ['immune cell'], 'ORGANISM': ['human ...","[human tonsils, .Human tonsils, Bovine Serum A...","['Homo sapiens', 'Bos taurus']",No,human,Homo sapiens,Yes
4,idr0149,Cohesin-mediated DNA loop extrusion resolves s...,protein localization ...,EFO,GO_0008104,Genetic information is stored in linear DNA mo...,Homo sapiens,2023-08-17,37357575,Cohesin-mediated DNA loop extrusion resolves s...,...,EFO_0003789 EFO_0003969 ...,To assess nuclear Sororin fluorescence in Soro...,Genetic information is stored in linear DNA mo...,"{'CELLULAR_COMPONENT': ['DNA', 'DNA', 'nuclear...",[human],['Homo sapiens'],No,human,Homo sapiens,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,idr0155,A new method for reliably detecting single flu...,protein localization,EFO,GO_0008104,Super-resolution and single-molecule microscop...,Homo sapiens,2024-03-05,,RASP: Optimal single puncta detection in compl...,...,EFO_0003969,Tissue sections were incubated with primary an...,Super-resolution and single-molecule microscop...,"{'CELL': ['cells', 'cell', 'cells', 'cellular'...","[human brain, patient]",['Homo sapiens'],No,mouse,Mus musculus,No
126,idr0044,In Toto Imaging and Reconstruction of Post-Imp...,time-lapse imaging,OMIT,OMIT_0027490,The mouse embryo has long been central to the ...,Mus musculus,2018-11-29,30318151,In Toto Imaging and Reconstruction of Post-Imp...,...,EFO_0001746 N/A,See attached methods N/A See attached methods ...,The mouse embryo has long been central to the ...,"{'ORGANISM': ['mouse embryo', 'mouse', 'mouse'...","[mouse embryo, mouse, mouse, mouse, E6.5]","['Mus musculus', 'Mus musculus', 'Mus musculus...",No,mouse,Mus musculus,Yes
127,idr0093,High cont

In [44]:
for row, i in dfs.iterrows():
    value_biobert = str(i['Specie_found_biobert'])
    

    if value_biobert.startswith("No"):
    
        message = [
        {"role": "system", "content": i["Description_combined"]},
        {"role": "user", "content": "Can you tell me which specie was used in this study?"},
        ]

        outputs = pipe(message, max_new_tokens = 256)
        response = outputs[0]["generated_text"][-1]["content"]
        dfs.at[row, 'Species_Llama'] = response
        print(f"Processed row {row} for species extraction.")

        if isinstance(i['Study_Organism'], str) and i['Study_Organism'].strip() != '':

            real_specie = i['Study_Organism'][0]

            if real_specie in response:
                print('The real species is in the response.')
                dfs.at[row, 'Species_found_Llama'] = 'Yes'
            else:
                print('The real species is NOT in the response.')
                dfs.at[row, 'Species_found_Llama'] = 'No'
        else:
            print('No real species to check in the response.')
            dfs.at[row, 'Species_found_Llama'] = 'No species provided'

       

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 0 for species extraction.
The real species is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 6 for species extraction.
The real species is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Processed row 124 for species extraction.
The real species is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 125 for species extraction.
The real species is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 128 for species extraction.
The real species is in the response.
Processed row 129 for species extraction.
The real species is in the response.


In [45]:
dfs['Species_found_Llama'].value_counts()

Species_found_Llama
Yes                    39
No                     17
No species provided     6
Name: count, dtype: int64

In [46]:
for row, i in dfs.iterrows():
    value_scipacy = str(i['Specie_found_biobert'])
    value_biobert = str(i['Specie_found_Scispacy'])
    value_lama = str(i['Species_found_Llama'])

    if value_scipacy.startswith("Yes") or value_biobert.startswith("Yes") or value_lama.startswith("Yes"):
        print(f"Row {row}: Species found in one of the methods.")
        dfs.at[row, 'Species_found'] = 'Yes'

    else:
        print(f"Row {row}: Species not found in any method.")
        dfs.at[row, 'Species_found'] = 'No'

Row 0: Species found in one of the methods.
Row 1: Species found in one of the methods.
Row 2: Species found in one of the methods.
Row 3: Species found in one of the methods.
Row 4: Species found in one of the methods.
Row 5: Species found in one of the methods.
Row 6: Species found in one of the methods.
Row 7: Species found in one of the methods.
Row 8: Species found in one of the methods.
Row 9: Species found in one of the methods.
Row 10: Species found in one of the methods.
Row 11: Species found in one of the methods.
Row 12: Species found in one of the methods.
Row 13: Species found in one of the methods.
Row 14: Species not found in any method.
Row 15: Species found in one of the methods.
Row 16: Species found in one of the methods.
Row 17: Species found in one of the methods.
Row 18: Species not found in any method.
Row 19: Species found in one of the methods.
Row 20: Species found in one of the methods.
Row 21: Species found in one of the methods.
Row 22: Species found in one

In [47]:
dfs['Species_found'].value_counts()

Species_found
Yes    107
No      23
Name: count, dtype: int64

In [56]:
for row, i in dfs.iterrows():
    value_scipacy = i['Specie_found_biobert']
    value_biobert = i['Specie_found_Scispacy']
    value_lama = i['Species_found_Llama']

    if value_scipacy.startswith("Yes") :
        print(f"Row {row}: Species found in SciSpacy and Llama, but not in BioBERT.")

Row 1: Species found in SciSpacy and Llama, but not in BioBERT.
Row 3: Species found in SciSpacy and Llama, but not in BioBERT.
Row 4: Species found in SciSpacy and Llama, but not in BioBERT.
Row 5: Species found in SciSpacy and Llama, but not in BioBERT.
Row 7: Species found in SciSpacy and Llama, but not in BioBERT.
Row 8: Species found in SciSpacy and Llama, but not in BioBERT.
Row 11: Species found in SciSpacy and Llama, but not in BioBERT.
Row 12: Species found in SciSpacy and Llama, but not in BioBERT.
Row 15: Species found in SciSpacy and Llama, but not in BioBERT.
Row 16: Species found in SciSpacy and Llama, but not in BioBERT.
Row 17: Species found in SciSpacy and Llama, but not in BioBERT.
Row 19: Species found in SciSpacy and Llama, but not in BioBERT.
Row 20: Species found in SciSpacy and Llama, but not in BioBERT.
Row 22: Species found in SciSpacy and Llama, but not in BioBERT.
Row 23: Species found in SciSpacy and Llama, but not in BioBERT.
Row 24: Species found in SciSpa